# Real LLM calls — the trick, step by step, against llama.cpp

**Platform: `llama-server` (llama.cpp).** That's what the repo serves on — `minijev.py` POSTs to `$DEP_LLAMA_URL/tokenize` and `/completion`, and `app.py` is a FastAPI queue on top. No ollama, no vLLM anywhere in this codebase.

Prereq: a server up with a chat model, e.g. `llama-server -m model.gguf -np 8`, then `DEP_LLAMA_URL=http://127.0.0.1:8080`.

Input is a **dict** — in production it's the parsed request JSON (via pydantic in `app.py`); here it's a plain dict. Every step prints everything: prompt, token ids, request body, raw numbers.

In [1]:
import json, os, sys, time
HERE = os.getcwd()
ROOT = os.path.dirname(HERE) if os.path.basename(HERE) == "learn" else HERE
sys.path.insert(0, ROOT)
# this machine: bonsai-2-27b on llama-server :17095, started with --api-key-file.
# minijev reads these at import time, so set them before the import.
os.environ.setdefault("DEP_LLAMA_URL", "http://127.0.0.1:17095")
os.environ.setdefault("DEP_API_KEY_FILE", "~/.claude/.secrets/bonsai-apikey.txt")
import minijev  # the REAL module, not a mimic

print("server:", minijev.URL)
t0 = time.perf_counter()
yes_id = minijev.token_id(" Yes")  # doubles as health check, like /api/health
print(f"server OK in {(time.perf_counter() - t0) * 1000:.0f} ms; ' Yes' -> token {yes_id}")


server: http://127.0.0.1:17095
server OK in 2 ms; ' Yes' -> token 7179


## Step 0 — input dict

Same shape as the README curl body, parsed. One `state`, N questions, each with `type` + `instructions` + `criteria`.

In [2]:
INPUT = {
    "state": "Sorry about the outage -- we have reset everyone's limits for the day.",
    "questions": {
        "is_quota_reset": {"type": "noul", "instructions": "Does this announce a quota reset?"},
        "urgency": {"type": "choice", "instructions": "How urgent is this?",
                    "criteria": {"ignore": "not relevant", "today": "act today",
                                 "now": "stop what you are doing"}},
    },
}
print(json.dumps(INPUT, indent=2))


{
  "state": "Sorry about the outage -- we have reset everyone's limits for the day.",
  "questions": {
    "is_quota_reset": {
      "type": "noul",
      "instructions": "Does this announce a quota reset?"
    },
    "urgency": {
      "type": "choice",
      "instructions": "How urgent is this?",
      "criteria": {
        "ignore": "not relevant",
        "today": "act today",
        "now": "stop what you are doing"
      }
    }
  }
}


## Step 1 — dict → prompt text (print the whole thing)

Same builders as `minijev.noul/choice`: menu of marks + one-token instruction, wrapped in `minijev._ASK` (user turn closed, assistant turn left open so the next token IS the answer).

In [3]:
state = INPUT["state"]
q = INPUT["questions"]["urgency"]
keys = list(q["criteria"])
marks = minijev._marks(len(keys))
menu = "\n".join(m + ". " + k + " -- " + q["criteria"][k] for m, k in zip(marks, keys))
body = "Text:\n" + state + "\n\nQuestion: " + q["instructions"] + "\nOptions:\n" + menu + f"\nReply with exactly one character: {', '.join(marks)}."
prompt = minijev._ASK.format(body=body)
labels = {k: " " + m for k, m in zip(keys, marks)}
print(prompt)
print("labels:", labels)


<|im_start|>user
Text:
Sorry about the outage -- we have reset everyone's limits for the day.

Question: How urgent is this?
Options:
1. ignore -- not relevant
2. today -- act today
3. now -- stop what you are doing
Reply with exactly one character: 1, 2, 3.<|im_end|>
<|im_start|>assistant
<think></think>Answer:
labels: {'ignore': ' 1', 'today': ' 2', 'now': ' 3'}


## Step 2 — labels → token ids (live `/tokenize`)

Each mark must be exactly one token. Watch the ids — you'll match them against the response next.

In [4]:
ids = {name: minijev.token_id(tok) for name, tok in labels.items()}
print(ids)  # e.g. {'ignore': 1234, 'today': 5678, 'now': 9012}


{'ignore': 16, 'today': 17, 'now': 18}


## Step 3 — request body + raw response (live `/completion`)

`n_predict=1`, equal `logit_bias` on candidates, `post_sampling_probs=True`, samplers off, `temperature=1.0`. Then: read `completion_probabilities[0].top_probs`, look up OUR ids, renormalise.

In [5]:
req_body = {
    "prompt": prompt,
    "n_predict": 1,
    "temperature": 1.0,
    "n_probs": min(40, max(20, len(ids) * 2)),
    "post_sampling_probs": True,
    "logit_bias": [[i, minijev.BIAS] for i in ids.values()],
    "top_k": 0, "top_p": 1.0, "min_p": 0.0, "typical_p": 1.0, "top_n_sigma": -1.0,
}
print(json.dumps({k: v for k, v in req_body.items() if k != "prompt"}, indent=2))

t0 = time.perf_counter()
out = minijev._post("/completion", req_body)
print(f"served in {(time.perf_counter() - t0) * 1000:.0f} ms")
top = {t["id"]: t["prob"] for t in out["completion_probabilities"][0].get("top_probs", [])}
print("candidate ids in report:", {n: (i, round(top.get(i, -1), 6)) for n, i in ids.items()})
raw = {name: top[i] for name, i in ids.items()}
z = sum(raw.values())
probs = {k: v / z for k, v in raw.items()}
print("renormalised:", {k: round(v, 4) for k, v in probs.items()})
print("pick:", max(probs, key=probs.get), "| confidence:", minijev.confidence(probs))


{
  "n_predict": 1,
  "temperature": 1.0,
  "n_probs": 20,
  "post_sampling_probs": true,
  "logit_bias": [
    [
      16,
      50.0
    ],
    [
      17,
      50.0
    ],
    [
      18,
      50.0
    ]
  ],
  "top_k": 0,
  "top_p": 1.0,
  "min_p": 0.0,
  "typical_p": 1.0,
  "top_n_sigma": -1.0
}
served in 149 ms
candidate ids in report: {'ignore': (16, 0.257308), 'today': (17, 0.620031), 'now': (18, 0.122661)}
renormalised: {'ignore': 0.2573, 'today': 0.62, 'now': 0.1227}
pick: today | confidence: 0.3627


## Step 4 — same thing in one line, both questions

Sanity check: `minijev` one-liners must agree with the manual numbers above.

In [6]:
p_noul, c_noul = minijev.noul(state, INPUT["questions"]["is_quota_reset"]["instructions"])
print("noul  : P(true) =", round(p_noul, 4), "confidence:", c_noul)
p_ch, c_ch = minijev.choice(state, q["instructions"], q["criteria"])
print("choice:", max(p_ch, key=p_ch.get), {k: round(v, 4) for k, v in p_ch.items()}, "confidence:", c_ch)


noul  : P(true) = 0.9839 confidence: 0.9678


choice: today {'ignore': 0.2573, 'today': 0.62, 'now': 0.1227} confidence: 0.3627
